In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets pillow gdown

In [ ]:
import os
import time
import torch
import pandas as pd

from PIL import Image

from datasets import Dataset

from transformers import (

    Qwen2_5_VLForConditionalGeneration,

    AutoProcessor,

    TrainingArguments,

    Trainer,

    BitsAndBytesConfig
)

from peft import (

    LoraConfig,

    get_peft_model
)

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    confusion_matrix,

    classification_report
)

In [ ]:
import gdown

folder_url = "https://drive.google.com/drive/folders/1hKLOtpVmF45IoBmJPwojgq6XraLtHmV6"

gdown.download_folder(

    url=folder_url,

    output="MultiOFF",

    quiet=False,

    remaining_ok=True
)

In [ ]:
DATASET_DIR = "./MultiOFF"

TRAIN_FILE = os.path.join(
    DATASET_DIR,
    "Split Dataset",
    "Training_meme_dataset.csv"
)

DEV_FILE = os.path.join(
    DATASET_DIR,
    "Split Dataset",
    "Validation_meme_dataset.csv"
)

TEST_FILE = os.path.join(
    DATASET_DIR,
    "Split Dataset",
    "Testing_meme_dataset.csv"
)

IMAGE_DIR = os.path.join(
    DATASET_DIR,
    "Labelled Images"
)

print(TRAIN_FILE)

print(DEV_FILE)

print(TEST_FILE)

print(IMAGE_DIR)

In [ ]:
print(os.path.exists(TRAIN_FILE))

print(os.path.exists(DEV_FILE))

print(os.path.exists(TEST_FILE))

print(os.path.exists(IMAGE_DIR))

In [ ]:
files = os.listdir(IMAGE_DIR)

print("NUMBER OF IMAGES:")

print(len(files))

print(files[:10])

In [ ]:
train_df = pd.read_csv(TRAIN_FILE)

dev_df = pd.read_csv(DEV_FILE)

test_df = pd.read_csv(TEST_FILE)

print(train_df.head())

print(train_df.columns)

In [ ]:
label_map = {

    "Non-offensiv": 0,

    "offensive": 1
}

train_df["label"] = train_df["label"].map(label_map)

dev_df["label"] = dev_df["label"].map(label_map)

test_df["label"] = test_df["label"].map(label_map)

print(train_df.head())

In [ ]:
def format_sample(row):

    image_path = os.path.join(
        IMAGE_DIR,
        str(row["image_name"])
    )

    if not os.path.exists(image_path):

        return None

    prompt = f"""
Analyze this meme.

Text:
{row['sentence']}

Classify it as:
- offensive
- non-offensive

Answer only one word.
"""

    return {

        "image": image_path,

        "prompt": prompt,

        "label": int(row["label"])
    }

In [ ]:
train_formatted = [

    format_sample(x)

    for _, x in train_df.iterrows()
]

dev_formatted = [

    format_sample(x)

    for _, x in dev_df.iterrows()
]

test_formatted = [

    format_sample(x)

    for _, x in test_df.iterrows()
]

train_formatted = [
    x for x in train_formatted
    if x is not None
]

dev_formatted = [
    x for x in dev_formatted
    if x is not None
]

test_formatted = [
    x for x in test_formatted
    if x is not None
]

print(len(train_formatted))

print(len(dev_formatted))

print(len(test_formatted))

In [ ]:
train_dataset = Dataset.from_list(
    train_formatted
)

dev_dataset = Dataset.from_list(
    dev_formatted
)

test_dataset = Dataset.from_list(
    test_formatted
)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

In [ ]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto",

    torch_dtype=torch.float16
)

In [ ]:
model.gradient_checkpointing_enable()

In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

In [ ]:
lora_config = LoraConfig(

    r=8,

    lora_alpha=16,

    target_modules=[

        "q_proj",

        "k_proj",

        "v_proj",

        "o_proj"
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

In [ ]:
def preprocess(example):

    image = Image.open(
        example["image"]
    ).convert("RGB")

    # MEMORY REDUCTION
    image = image.resize((224, 224))

    messages = [
        {
            "role": "user",

            "content": [

                {
                    "type": "image",
                    "image": image
                },

                {
                    "type": "text",
                    "text": example["prompt"]
                }
            ]
        }
    ]

    text = processor.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True
    )

    inputs = processor(

        text=[text],

        images=[image],

        max_length=256,

        truncation=True,

        return_tensors="pt"
    )

    answer_text = (

        "offensive"

        if example["label"] == 1

        else "non-offensive"
    )

    answer_tokens = processor.tokenizer(

        answer_text,

        add_special_tokens=False

    ).input_ids

    input_ids = inputs[
        "input_ids"
    ][0]

    attention_mask = inputs[
        "attention_mask"
    ][0]

    labels = torch.cat([

        torch.full(
            (len(input_ids),),
            -100,
            dtype=torch.long
        ),

        torch.tensor(
            answer_tokens,
            dtype=torch.long
        )
    ])

    input_ids = torch.cat([

        input_ids,

        torch.tensor(
            answer_tokens,
            dtype=torch.long
        )
    ])

    attention_mask = torch.cat([

        attention_mask,

        torch.ones(
            len(answer_tokens),
            dtype=torch.long
        )
    ])

    return {

        "input_ids": input_ids,

        "attention_mask": attention_mask,

        "labels": labels,

        "pixel_values": inputs[
            "pixel_values"
        ],

        "image_grid_thw": inputs[
            "image_grid_thw"
        ]
    }

In [ ]:
train_dataset = train_dataset.map(

    preprocess,

    load_from_cache_file=False
)

dev_dataset = dev_dataset.map(

    preprocess,

    load_from_cache_file=False
)

test_dataset = test_dataset.map(

    preprocess,

    load_from_cache_file=False
)

In [ ]:
train_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "labels",

        "pixel_values",

        "image_grid_thw"
    ]
)

dev_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "labels",

        "pixel_values",

        "image_grid_thw"
    ]
)

test_dataset.set_format(

    type="torch",

    columns=[

        "input_ids",

        "attention_mask",

        "labels",

        "pixel_values",

        "image_grid_thw"
    ]
)

In [ ]:
sample = train_dataset[0]

print(sample.keys())

print(sample["pixel_values"].shape)

print(sample["image_grid_thw"].shape)

In [ ]:
def collate_fn(batch):

    input_ids = torch.nn.utils.rnn.pad_sequence(

        [x["input_ids"] for x in batch],

        batch_first=True,

        padding_value=processor.tokenizer.pad_token_id
    )

    attention_mask = torch.nn.utils.rnn.pad_sequence(

        [x["attention_mask"] for x in batch],

        batch_first=True,

        padding_value=0
    )

    labels = torch.nn.utils.rnn.pad_sequence(

        [x["labels"] for x in batch],

        batch_first=True,

        padding_value=-100
    )

    pixel_values = torch.cat([

        x["pixel_values"]

        for x in batch

    ], dim=0)

    image_grid_thw = torch.cat([

        x["image_grid_thw"]

        for x in batch

    ], dim=0)

    return {

        "input_ids": input_ids,

        "attention_mask": attention_mask,

        "labels": labels,

        "pixel_values": pixel_values,

        "image_grid_thw": image_grid_thw
    }

In [ ]:
training_args = TrainingArguments(

    output_dir="./qwen_hatememes",

    per_device_train_batch_size=1,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=1,

    num_train_epochs=30,

    learning_rate=2e-4,

    logging_steps=10,

    save_strategy="epoch",

    eval_strategy="epoch",

    remove_unused_columns=False,

    fp16=True,

    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=dev_dataset,

    data_collator=collate_fn
)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
start_train_time = time.time()

trainer.train()

end_train_time = time.time()

training_time = (
    end_train_time -
    start_train_time
)

hours = int(training_time // 3600)

minutes = int(
    (training_time % 3600) // 60
)

seconds = int(
    training_time % 60
)

print(

    f"Training Time: "

    f"{hours}h "

    f"{minutes}m "

    f"{seconds}s"
)

In [ ]:
def predict(sample):

    image = Image.open(
        sample["image"]
    ).convert("RGB")

    image = image.resize((224, 224))

    messages = [
        {
            "role": "user",

            "content": [

                {
                    "type": "image",
                    "image": image
                },

                {
                    "type": "text",
                    "text": sample["prompt"]
                }
            ]
        }
    ]

    text = processor.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True
    )

    inputs = processor(

        text=[text],

        images=[image],

        return_tensors="pt"
    )

    inputs = {

        k: v.to(model.device)

        for k, v in inputs.items()
    }

    with torch.no_grad():

        output = model.generate(

            **inputs,

            max_new_tokens=5
        )

    decoded = processor.batch_decode(

        output,

        skip_special_tokens=True
    )[0]

    decoded = decoded.lower()

    if "offensive" in decoded:

        return 1

    return 0

In [ ]:
y_true = []

y_pred = []

inference_times = []

for sample in test_formatted:

    torch.cuda.synchronize()

    start_time = time.time()

    prediction = predict(sample)

    torch.cuda.synchronize()

    end_time = time.time()

    inference_times.append(
        end_time - start_time
    )

    y_true.append(
        sample["label"]
    )

    y_pred.append(
        prediction
    )

accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred
)

recall = recall_score(
    y_true,
    y_pred
)

f1 = f1_score(
    y_true,
    y_pred
)

cm = confusion_matrix(
    y_true,
    y_pred
)

total_inference_time = sum(
    inference_times
)

average_inference_time = (

    total_inference_time /

    len(inference_times)
)

print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("F1-score:", f1)

print()

print("Confusion Matrix:")

print(cm)

print()

print(

    f"Total Inference Time: "

    f"{total_inference_time:.4f} seconds"
)

print(

    f"Average Time Per Sample: "

    f"{average_inference_time:.6f} seconds"
)

In [ ]:
report = classification_report(

    y_true,

    y_pred,

    target_names=[
        "Non-Offensive",
        "Offensive"
    ]
)
print(report)